# YouTube Toxic Comment Classification

## Part 1 — Introduction

### Student details

- IdoM — ID ending: [8349]
- ShirZ — ID ending: [4811]
- RoeeS — ID ending: [5498]

### AI prompts and additional resources

| Tool or resource | Prompt | Purpose |
|---|---|---|
| ChatGPT / Codex | "Take a look at the assignment and explain what is required." | Understanding the assignment requirements. |
| ChatGPT / Codex | "Which algorithm options are suitable for this classification problem?" | Selecting a learning algorithm. |
| ChatGPT / Codex | "Do we need to define quality metrics?" | Understanding the required evaluation metric. |
| Kaggle | https://www.kaggle.com/datasets/reihanenamdari/youtube-toxicity-data | Dataset source. |

### Learning problem and dataset

This project addresses a supervised binary text-classification problem: predicting whether an English YouTube comment is toxic. The input is the comment from the `Text` column, and the target is `IsToxic`, where `TRUE` represents a toxic comment and `FALSE` represents a non-toxic comment. The selected Kaggle dataset contains 1,000 manually labelled YouTube comments and additional labels describing different toxicity categories.

In [ ]:
import pandas as pd

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [ ]:
train_df.head(5)

In [ ]:
test_df.head(5)

# Quality metric

This is a binary classification problem with one central class: toxic comments. Therefore, according to the assignment instructions, model quality will be evaluated using the F1 score for the toxic class only (`IsToxic = 1`).

The F1 score combines precision and recall. Precision measures how many of the comments predicted as toxic are actually toxic, while recall measures how many of the truly toxic comments were correctly identified. This is appropriate for the current task because both failing to detect a toxic comment and incorrectly flagging a non-toxic comment are important errors.

The F1 score is calculated as:

$$
F1 = 2 \cdot \frac{\mathrm{Precision} \cdot \mathrm{Recall}}{\mathrm{Precision} + \mathrm{Recall}}
$$

The same metric will be used throughout cross-validation, hyperparameter selection, and final test-set evaluation.

In [ ]:
from sklearn.metrics import f1_score

def calculate_quality(y_true, y_pred):
    """Calculate the F1 score for the toxic class."""
    return f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

# Part 2 — Feature Engineering

## Text preprocessing

Before extracting features, line breaks, non-breaking spaces, and repeated whitespace are replaced with a single regular space, and the text is converted to lowercase. The same preprocessing is applied to the train and test sets.

In [ ]:
import re

def preprocess_text(text):
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

train_df["CleanText"] = train_df["Text"].apply(preprocess_text)
test_df["CleanText"] = test_df["Text"].apply(preprocess_text)

### Preprocessing examples from the train set

In [ ]:
train_df[["Text", "CleanText"]].head(3)

### Preprocessing examples from the test set

In [ ]:
test_df[["Text", "CleanText"]].head(3)

## TF-IDF feature extraction

TF-IDF converts every cleaned comment into a numerical feature vector. A word receives a higher weight when it is frequent in a particular comment but less frequent across the full training set. The vectorizer is fitted only on the train set. The fitted vectorizer is then used to transform the test set without learning information from it.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df["CleanText"])
X_test_tfidf = tfidf_vectorizer.transform(test_df["CleanText"])

print("Train feature matrix shape:", X_train_tfidf.shape)
print("Test feature matrix shape:", X_test_tfidf.shape)

### TF-IDF examples

For each example, the following function displays the cleaned comment and its ten features with the highest non-zero TF-IDF weights.

In [ ]:
def show_tfidf_examples(clean_texts, feature_matrix, vectorizer, number_of_examples=3):
    feature_names = vectorizer.get_feature_names_out()

    for position in range(min(number_of_examples, feature_matrix.shape[0])):
        row = feature_matrix.getrow(position)
        ordered_positions = row.data.argsort()[::-1][:10]
        features = pd.DataFrame({
            "feature": feature_names[row.indices[ordered_positions]],
            "tfidf_weight": row.data[ordered_positions],
        })

        print(f"Example {position + 1}: {clean_texts.iloc[position]}")
        display(features)

### Three train-set examples

In [ ]:
show_tfidf_examples(
    train_df["CleanText"],
    X_train_tfidf,
    tfidf_vectorizer,
)

### Three test-set examples

In [ ]:
show_tfidf_examples(
    test_df["CleanText"],
    X_test_tfidf,
    tfidf_vectorizer,
)